In [1]:
# Importing All Necessary Libraries
import pandas as pd
import numpy as np
import random
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

In the seed code below I have tried to keep the performance of the model same by removing the randomness and fixing the seed. So that the result in my report and on running the code matches.

In [3]:
# Setting Random Seeds for Reproducible same robust results

seed_value = 50
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_value)

In [ ]:
df = pd.read_csv('D:/CODING/projects GitHub/tweet_sentiment_analysis/dataset/1.6million_tweets/sentiment140_cleaned.csv', encoding='latin-1')
print(f"Successfully loaded manually corrected dataset with {len(df)} tweets.")

Successfully loaded manually corrected dataset with 1600000 tweets.


In [5]:
df.head()

,target,id,date,flag,user,text,cleaned_text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot httptwitpiccom2y1zl awww thats bumm...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset cant update facebook texting might cry r...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dived many times ball managed save 50...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole body feels itchy like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behaving im mad cant see


In [6]:
df['cleaned_text'] = df['cleaned_text'].fillna(df['text'])  # Use original text if cleaned_text is null

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 7 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   target        1600000 non-null  int64 
 1   id            1600000 non-null  int64 
 2   date          1600000 non-null  object
 3   flag          1600000 non-null  object
 4   user          1600000 non-null  object
 5   text          1600000 non-null  object
 6   cleaned_text  1600000 non-null  object
dtypes: int64(2), object(5)
memory usage: 85.4+ MB


In [8]:
df['cleaned_text'] = df['cleaned_text'].astype(str)

In [9]:
# # Preprocessing the text data

# def preprocess_text(text):
#     text = str(text)  # Converts the input to a string in case it is not already one
#     text = re.sub(r'http\S+|www\S+', ' ', text)  # Removes URLs starting with http, https, or www
#     text = re.sub(r'@\w+', ' @user ', text)  # Replaces @mentions with a generic placeholder token
#     text = re.sub(r'#(\w+)', r'\1', text)  # Removes the # symbol but keeps the hashtag word itself
#     text = re.sub(r'\s+', ' ', text).strip()  # Collapses multiple spaces into one and trims leading/trailing spaces
#     return text
# df['cleaned_text'] = df['text'].apply(preprocess_text)
# print("Preprocessing complete.")

In [10]:
df['target'].value_counts()

,count
target,
0,800000
1,800000


In [11]:
# Preparing data for the model for training
texts = df['cleaned_text'].tolist() #sepearting texts
labels = df['target'].tolist() #seperating the classification data

# splitting the data into a training set and a testing set.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print("Data has been split.")

Data has been split.


In [12]:
#This guarantees labels are numeric integers before dataset creation and training.
train_labels = [int(x) for x in train_labels]
test_labels = [int(x) for x in test_labels]

In [13]:
# Tokenization and padding
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# here the Hugging Face tokenizer is tokenizing, truncating and padding
# USING TOO MUCH RAM Hence the below change
"""train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=64)
print("Tokenization and padding of data complete.")"""

# Regulate max_length of tweets tokenization to reduce RAM usage
max_length = 128

print("Tokenizer loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded successfully.


In [14]:
#This tokenizes each sample only when needed during training, instead of storing all tokenized rows in RAM.
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_length=max_length)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, max_length=max_length)

#can tweak batch_size for RAM usage moderation
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("Created PyTorch datasets and dataloaders successfully.")

Created PyTorch datasets and dataloaders successfully.


In [15]:
# Loading the Pre-Trained Transformer model DistilBERT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print("Pre-trained DistilBERT PyTorch model loaded successfully.")
print(f"Using device: {device}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Pre-trained DistilBERT PyTorch model loaded successfully.
Using device: cuda


In [16]:
# Training the model

# using EarlyStopping to prevent overfitting.
from copy import deepcopy

num_epochs = 10
patience = 1
best_val_loss = float('inf')
patience_counter = 0

history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

best_model_state = None

from tqdm.auto import tqdm
from copy import deepcopy

num_epochs = 10
patience = 1
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        avg_loss = total_train_loss / step

        progress_bar.set_postfix(train_loss=f"{avg_loss:.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    correct = 0
    total = 0

    val_bar = tqdm(test_loader, desc=f"Validating {epoch+1}/{num_epochs}", leave=False)
    with torch.no_grad():
        for batch in val_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch['labels']).sum().item()
            total += batch['labels'].size(0)

    avg_val_loss = total_val_loss / len(test_loader)
    val_accuracy = correct / total

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(val_accuracy)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter > patience:
            print("Early stopping triggered.")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("\nModel fine-tuning complete.")

Epoch 1/10:   0%|          | 0/80000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.eval()
all_preds = []
all_labels = []
total_test_loss = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)

        total_test_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

avg_test_loss = total_test_loss / len(test_loader)
test_accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

print(f"Final Test Loss: {avg_test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Visualizing the learning process (train and test curves)

print("\nVisualizing Training History...")
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['val_accuracy'])
ax1.set_title('Validation Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend(['Validation'], loc='upper left')

ax2.plot(history['train_loss'])
ax2.plot(history['val_loss'])
ax2.set_title('Model Loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Creating a Prediction function for new inputs

def predict_sentiment_transformer(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length', max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class_id = int(torch.argmax(outputs.logits, dim=-1)[0].cpu().item())

    return "Positive" if predicted_class_id == 1 else "Negative"

In [ ]:
# Prediction user interface (CLI)
while True:
    user_input = input("Enter a sentence to analyze, or type 'quit' to exit:")
    if user_input.lower() == 'quit':
        break
    #calling the prediction function
    sentiment = predict_sentiment_transformer(user_input)
    print(f"Predicted Sentiment: {sentiment}")